In [1]:
import numpy as np
from matplotlib import pyplot as plt
from pathlib import Path

from pydrake.all import (
    AddMultibodyPlantSceneGraph,
    AngleAxis,
    DiagramBuilder,
    Integrator,
    JacobianWrtVariable,
    LeafSystem,
    MeshcatVisualizer,
    MultibodyPlant,
    MultibodyPositionToGeometryPose,
    Parser,
    PiecewisePolynomial,
    PiecewisePose,
    PiecewiseQuaternionSlerp,
    Quaternion,
    Rgba,
    RigidTransform,
    RotationMatrix,
    SceneGraph,
    Simulator,
    StartMeshcat,
    TrajectorySource,
    StateInterpolatorWithDiscreteDerivative,
    Multiplexer,
    Adder,
    LogVectorOutput,
    InverseDynamicsController,
    PidController,
    Trajectory,
    ApplyMultibodyPlantConfig,
    ModelDirectives,
    ProcessModelDirectives,
    VisualizationConfig,
    ApplyVisualizationConfig,
    ApplyLcmBusConfig,
    LcmSubscriberSystem,
    Value,
    Context,
    BasicVector,
    LcmPublisherSystem,
)

from manipulation import running_as_notebook
from manipulation.station import LoadScenario, MakeHardwareStation
from manipulation.utils import RenderDiagram, ConfigureParser

import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
from lcmdefs.messages.so101 import lcmt_so101_configuration

In [2]:
meshcat = StartMeshcat()

INFO:drake:Meshcat listening for connections at http://localhost:7000


## Define Diff-IK Controller

In [3]:
class PseudoInverseController(LeafSystem):
    def __init__(self, plant: MultibodyPlant):
        LeafSystem.__init__(self)
        self._plant = plant
        self._plant_context = plant.CreateDefaultContext()
        self._so101 = plant.GetModelInstanceByName("so101")
        self._G = plant.GetBodyByName("gripper_link", self._so101).body_frame()
        self._W = plant.world_frame()

        self.V_WG_port = self.DeclareVectorInputPort("V_WG", 6)
        self.x_port = self.DeclareVectorInputPort("so101.state", 12)
        self.DeclareVectorOutputPort("so101.velocity", 6, self.CalcOutput)

    def CalcOutput(self, context, output):
        V_WG_desired = self.V_WG_port.Eval(context)
        x = self.x_port.Eval(context)
        self._plant.SetPositionsAndVelocities(self._plant_context, self._so101, x)
        J_G = self._plant.CalcJacobianSpatialVelocity(
            self._plant_context,
            JacobianWrtVariable.kQDot,
            self._G,
            [0, 0, 0],
            self._W,
            self._W,
        )
        J_G = J_G[:, :5]  # Ignore gripper term

        v = np.linalg.pinv(J_G).dot(V_WG_desired)
        v = np.append(v, 0)
        # V_G_actual = J_G.dot(v)
        # print(f"V_G desired: {V_G_desired}, actual: {V_G_actual}")
        output.SetFromVector(v)

## Define Trajectories

In [6]:
event_deltas = [
    ('initial_rest', 1.0),
    ('initial_ready', 3.0),
    ('prepick', 2.0),
    ('pick_opened', 1.0),
    ('pick_closed', 1.0),
    ('postpick', 1.0),
    ('midpoint', 2.0),
    ('preplace', 2.0),
    ('place_closed', 1.0),
    ('place_opened', 1.0),
    ('place_moveaway', 0.5),
    ('postplace', 1.0),
    ('final_ready', 2.0),
    ('final_rest', 3.0),
]

hold_time = 1.0
raw_names, event_names, event_times = [], ['initial'], [0.0]
for name, delta in event_deltas:
    raw_names.append(name)
    event_names.append(name + "_start")
    event_times.append(event_times[-1] + delta)
    event_names.append(name + "_end")
    event_times.append(event_times[-1] + hold_time)
events = {n: t for n, t in zip(event_names, event_times)}
num_events = len(event_names)


def GetEventValues(value_dict):
    first_value = None
    for raw_name in raw_names:
        if raw_name in value_dict:
            first_value = value_dict[raw_name]
            break
    event_values = [first_value]
    current_value = first_value
    for raw_name in raw_names:
        if raw_name in value_dict:
            current_value = value_dict[raw_name]
        event_values.extend([current_value] * 2)
    return event_values

def MakeGripperTrajectory(
    X_Ginitial: RigidTransform,
    X_Oinitial: RigidTransform,
    X_Ogoal: RigidTransform,   
) -> PiecewisePose:
    X_G = {
        'initial_ready': X_Ginitial,
        'midpoint': X_Ginitial,
        'final_ready': X_Ginitial,
    }

    p_GgraspO = [0.02, 0, -0.12]
    R_GgraspO = RotationMatrix.MakeZRotation(np.pi / 2)
    X_GgraspO = RigidTransform(R_GgraspO, p_GgraspO)
    X_OGgrasp = X_GgraspO.inverse()
    X_Gpick = X_Oinitial @ X_OGgrasp
    X_Gplace = X_Ogoal @ X_OGgrasp
    X_G["pick_opened"] = X_Gpick
    X_G["pick_closed"] = X_Gpick
    X_G["place_closed"] = X_Gplace
    X_G["place_opened"] = X_Gplace

    p_GplaceGmoveaway = [-0.01, 0, 0]
    X_GplaceGmoveaway = RigidTransform(RotationMatrix.Identity(), p_GplaceGmoveaway)
    X_Gmoveaway = X_Gplace @ X_GplaceGmoveaway
    X_G["place_moveaway"] = X_Gmoveaway

    X_GgraspGpregrasp = RigidTransform([0, 0, 0.05])
    X_Gprepick = X_Gpick @ X_GgraspGpregrasp
    X_G["prepick"] = X_Gprepick
    X_G["postpick"] = X_Gprepick
    X_G["preplace"] = X_Gplace @ X_GgraspGpregrasp
    X_G["postplace"] = X_Gmoveaway @ X_GgraspGpregrasp

    event_values = GetEventValues(X_G)
    return PiecewisePose.MakeLinear(event_times, event_values)
    
def MakeJointTrajectory() -> PiecewisePolynomial:
    q_rest = np.array([0, -1.822, 1.572, 0.906, 0, 0])
    q_ready = np.array([0, 0, 0, 1.5, 0, 0])
    finger_closed = q_ready + np.array([0, 0, 0, 0, 0, 0.1])
    finger_opened = q_ready + np.array([0, 0, 0, 0, 0, 0.5])
    
    q = {
        'initial_rest': q_rest,
        'initial_ready': q_ready,
        'prepick': finger_opened,
        'pick_closed': finger_closed,
        'place_opened': finger_opened,
        'final_ready': q_ready,
        'final_rest': q_rest,
    }

    event_values = GetEventValues(q)
    return PiecewisePolynomial.FirstOrderHold(event_times, np.array(event_values).T)


X_Oinitial = RigidTransform(RotationMatrix.MakeZRotation(np.pi / 2), [-0.075, 0.025, 0.05])
X_Ogoal = RigidTransform(RotationMatrix.MakeZRotation(-np.pi / 2), [0.075, 0.025, 0])

## Simulate Task

In [7]:
builder = DiagramBuilder()

scenario = LoadScenario(filename="../scenarios/so101_block.yaml")
station = builder.AddSystem(MakeHardwareStation(scenario, meshcat=meshcat))
plant: MultibodyPlant = station.GetSubsystemByName("plant")

brick_model = plant.GetModelInstanceByName("box")
brick_body = plant.GetRigidBodyByName("box_link", brick_model)
plant.SetDefaultFloatingBaseBodyPose(brick_body, X_Oinitial)

so101_model = plant.GetModelInstanceByName("so101")
plant.SetDefaultPositions(so101_model, np.array([0, 0, 0, 1.5, 0, 0]))
so101_gripper_body = plant.GetRigidBodyByName("gripper_link", so101_model)
temp_context = station.CreateDefaultContext()
temp_plant_context = plant.GetMyContextFromRoot(temp_context)
X_Ginitial = plant.EvalBodyPoseInWorld(temp_plant_context, so101_gripper_body)
X_Oinitial.set_translation([-0.075, 0.025, 0])
traj_X_G = MakeGripperTrajectory(X_Ginitial, X_Oinitial, X_Ogoal)
traj_V_G = traj_X_G.MakeDerivative()
V_G_source = builder.AddSystem(TrajectorySource(traj_V_G))

q_rest = np.array([0, -1.822, 1.572, 0.906, 0, 0])
plant.SetDefaultPositions(so101_model, q_rest)
traj_q: PiecewisePolynomial = MakeJointTrajectory()
traj_qdot = traj_q.MakeDerivative()
qdot_source = builder.AddNamedSystem("qdot_source", TrajectorySource(traj_qdot))

controller = builder.AddSystem(PseudoInverseController(plant))
adder = builder.AddSystem(Adder(2, 6))
integrator = builder.AddSystem(Integrator(6))
interpolator = builder.AddSystem(StateInterpolatorWithDiscreteDerivative(6, 1e-4))
builder.Connect(
    controller.get_output_port(),
    adder.GetInputPort("u0")
)
builder.Connect(
    qdot_source.get_output_port(),
    adder.GetInputPort("u1")
)
builder.Connect(
    adder.get_output_port(),
    integrator.get_input_port()
)
builder.Connect(
    integrator.get_output_port(),
    interpolator.get_input_port()
)
builder.Connect(
    interpolator.get_output_port(),
    station.GetInputPort("so101.desired_state")
)
builder.Connect(
    station.GetOutputPort("so101_state"),
    controller.GetInputPort("so101.state")
)
builder.Connect(
    V_G_source.get_output_port(),
    controller.GetInputPort("V_WG")
)

diagram = builder.Build()
# RenderDiagram(diagram)

simulator = Simulator(diagram)
context = simulator.get_mutable_context()
integrator.set_integral_value(
    integrator.GetMyContextFromRoot(context),
    plant.GetPositions(
        plant.GetMyContextFromRoot(context),
        plant.GetModelInstanceByName("so101"),
    ),
)

meshcat.StartRecording()
simulator.AdvanceTo(traj_V_G.end_time())
meshcat.StopRecording()
meshcat.PublishRecording()

## Perform Task On Hardware

In [8]:
class SO101StatusReceiver(LeafSystem):
    def __init__(self):
        super().__init__()

        self.input_port = self.DeclareAbstractInputPort(
            name="lcmt_so101_configuration",
            model_value=Value(lcmt_so101_configuration())
        )
        self.DeclareVectorOutputPort(
            name="position_measured",
            size=6,
            calc=self.ParseObservation
        )

    def ParseObservation(self, context: Context, output: BasicVector) -> None:
        observation: lcmt_so101_configuration = self.input_port.Eval(context)
        q_current = np.array(observation.q)
        output.SetFromVector(q_current)

class SO101CommandSender(LeafSystem):
    def __init__(self):
        super().__init__()

        self.input_port = self.DeclareVectorInputPort(
            name="position", 
            size=6
        )
        self.DeclareAbstractOutputPort(
            name="lcmt_command",
            alloc=lambda: Value(lcmt_so101_configuration()),
            calc=self.FormAction
        )

    def FormAction(self, context: Context, output):
        q = self.input_port.Eval(context)
        action = lcmt_so101_configuration()
        action.q = q
        output.set_value(action)

In [9]:
scenario = LoadScenario(filename="../scenarios/so101_hardware.yaml")
builder = DiagramBuilder()
scene_graph = SceneGraph()
builder.AddNamedSystem("scene_graph", scene_graph)
plant = MultibodyPlant(time_step=scenario.plant_config.time_step)
ApplyMultibodyPlantConfig(scenario.plant_config, plant)
plant.RegisterAsSourceForSceneGraph(scene_graph)
parser = Parser(plant)
ConfigureParser(parser)

added_models = ProcessModelDirectives(
    directives=ModelDirectives(directives=scenario.directives),
    parser=parser,
)

plant.Finalize()

to_pose = MultibodyPositionToGeometryPose(plant)
builder.AddSystem(to_pose)
builder.Connect(
    to_pose.get_output_port(),
    scene_graph.get_source_pose_port(plant.get_source_id()),
)

config = VisualizationConfig()
config.publish_contacts = False
config.publish_inertia = False
ApplyVisualizationConfig(
    config,
    builder=builder,
    plant=plant,
    scene_graph=scene_graph,
    meshcat=meshcat,
)

lcm_buses = ApplyLcmBusConfig(lcm_buses=scenario.lcm_buses, builder=builder)

lcm = lcm_buses.Find("SO101 Bus", "so101_lcm")

publish_period = 0.005
so101_command_sender = SO101CommandSender()
builder.AddNamedSystem("so101.command_sender", so101_command_sender)
so101_command_publisher = LcmPublisherSystem.Make(
    channel="SO101_COMMAND",
    lcm_type=lcmt_so101_configuration,
    lcm=lcm,
    publish_period=publish_period,
    use_cpp_serializer=False,
)
builder.AddNamedSystem("so101.command_publisher", so101_command_publisher)
builder.Connect(
    so101_command_sender.get_output_port(),
    so101_command_publisher.get_input_port()
)

so101_status_receiver = SO101StatusReceiver()
builder.AddNamedSystem("so101.status_receiver", so101_status_receiver)
so101_status_subscriber = LcmSubscriberSystem.Make(
    channel="SO101_STATUS",
    lcm_type=lcmt_so101_configuration,
    lcm=lcm,
    use_cpp_serializer=False,
    wait_for_message_on_initialization_timeout=10,
)
builder.AddNamedSystem("so101.status_subscriber", so101_status_subscriber)

builder.ExportOutput(
    so101_status_receiver.get_output_port(),
    "so101.position_measured"
)

obs_interpolator = StateInterpolatorWithDiscreteDerivative(6, 1e-4)
builder.AddNamedSystem("so101.state_interpolator", obs_interpolator)
builder.Connect(
    so101_status_receiver.get_output_port(),
    obs_interpolator.get_input_port()
)

builder.Connect(
    so101_status_subscriber.get_output_port(),
    so101_status_receiver.get_input_port()
)
builder.Connect(
    so101_status_receiver.get_output_port(),
    to_pose.get_input_port()
)

so101_model = plant.GetModelInstanceByName("so101")
plant.SetDefaultPositions(so101_model, np.array([0, 0, 0, 1.5, 0, 0]))
so101_gripper_body = plant.GetRigidBodyByName("gripper_link", so101_model)
temp_plant_context = plant.CreateDefaultContext()
X_Ginitial = plant.EvalBodyPoseInWorld(temp_plant_context, so101_gripper_body)
X_Oinitial.set_translation([-0.075, 0.025, 0])
traj_X_G = MakeGripperTrajectory(X_Ginitial, X_Oinitial, X_Ogoal)
traj_V_G = traj_X_G.MakeDerivative()
V_G_source = builder.AddSystem(TrajectorySource(traj_V_G))

q_rest = np.array([0, -1.822, 1.572, 0.906, 0, 0])
plant.SetDefaultPositions(so101_model, q_rest)
traj_q: PiecewisePolynomial = MakeJointTrajectory()
traj_qdot = traj_q.MakeDerivative()
qdot_source = builder.AddNamedSystem("qdot_source", TrajectorySource(traj_qdot))

controller = builder.AddSystem(PseudoInverseController(plant))
adder = builder.AddSystem(Adder(2, 6))
integrator = builder.AddSystem(Integrator(6))
builder.Connect(
    controller.get_output_port(),
    adder.GetInputPort("u0")
)
builder.Connect(
    qdot_source.get_output_port(),
    adder.GetInputPort("u1")
)
builder.Connect(
    adder.get_output_port(),
    integrator.get_input_port()
)
builder.Connect(
    integrator.get_output_port(),
    so101_command_sender.get_input_port()
)
builder.Connect(
    obs_interpolator.get_output_port(),
    controller.GetInputPort("so101.state")
)
builder.Connect(
    V_G_source.get_output_port(),
    controller.GetInputPort("V_WG")
)

diagram = builder.Build()
diagram.set_name("SO101StationInterface")

# RenderDiagram(diagram)

simulator = Simulator(diagram)
simulator.set_target_realtime_rate(1.0)

context = simulator.get_mutable_context()
integrator.set_integral_value(
    integrator.GetMyContextFromRoot(context),
    q_rest
)

meshcat.StartRecording()
simulator.AdvanceTo(traj_V_G.end_time(), interruptible=True)
meshcat.StopRecording()
meshcat.PublishRecording()

INFO:drake:LCM bus 'so101_lcm' created for URL udpm://239.255.76.67:7667?ttl=0
==== LCM Warning ===
LCM detected that large packets are being received, but the kernel UDP
receive buffer is very small.  The possibility of dropping packets due to
insufficient buffer space is very high.

For more information, visit:
   https://lcm-proj.github.io/lcm/content/multicast-setup.html

==== LCM Warning ===
LCM detected that large packets are being received, but the kernel UDP
receive buffer is very small.  The possibility of dropping packets due to
insufficient buffer space is very high.

For more information, visit:
   https://lcm-proj.github.io/lcm/content/multicast-setup.html

